# **1. Parâmetros e cadastro dos módulos**


Define o limiar mínimo de combustível para pousar (variável C = 30%) e cadastra os 5 módulos da colônia. Cada módulo é um dicionário com prioridade, combustível, massa, criticidade da carga e horário de chegada. Todos ficam em uma lista.

In [2]:
LIMIAR_COMBUSTIVEL = 30  # % mínimo de combustível para pousar (variável C)

modulos = [
    {"nome": "Energia",        "prioridade": 5, "combustivel": 70, "massa": 8000,  "criticidade": "Alta",    "chegada": "08:10"},
    {"nome": "Habitação",      "prioridade": 4, "combustivel": 65, "massa": 12000, "criticidade": "Alta",    "chegada": "08:30"},
    {"nome": "Suporte Médico", "prioridade": 5, "combustivel": 45, "massa": 5000,  "criticidade": "Crítica", "chegada": "08:45"},
    {"nome": "Laboratório",    "prioridade": 3, "combustivel": 80, "massa": 7000,  "criticidade": "Média",   "chegada": "09:00"},
    {"nome": "Logística",      "prioridade": 2, "combustivel": 90, "massa": 9000,  "criticidade": "Baixa",   "chegada": "09:20"},
]

# **2. Estruturas de dados lineares**

Cria as estruturas que organizam a operação:

Fila (fila_pouso): módulos aguardando autorização, com entrada no fim e saída pelo início (FIFO).
Listas (pousados, em_espera, em_alerta): módulos já pousados, adiados e em emergência.
Pilha (historico): registra os eventos, e o último a entrar é o primeiro a sair (LIFO).

In [ ]:
fila_pouso = []   # FILA: módulos aguardando autorização
pousados = []     # LISTA: módulos que já pousaram
em_espera = []    # LISTA: módulos adiados
em_alerta = []    # LISTA: módulos em alerta/emergência
historico = []    # PILHA: eventos (último a entrar é o primeiro a sair)

# **3. Algoritmos de ordenação**

Define o critério de prioridade de pouso: maior prioridade primeiro, depois menor combustível, depois menor horário de chegada. Implementa dois algoritmos:

Ordenação por inserção: monta a fila de pouso.
Ordenação por seleção: deixa a lista em ordem crescente de prioridade, como a busca binária exige.

In [ ]:
def chave(modulo):
    """Critério: maior prioridade primeiro (sinal negativo), depois
    menor combustível, depois horário de chegada."""
    return (-modulo["prioridade"], modulo["combustivel"], modulo["chegada"])


def ordenar_insercao(lista):
    """Ordenação por inserção usando o critério 'chave'."""
    for i in range(1, len(lista)):
        guardado = lista[i]
        j = i - 1
        # empurra para a direita os itens "maiores" que o guardado
        while j >= 0 and chave(lista[j]) > chave(guardado):
            lista[j + 1] = lista[j]
            j -= 1
        lista[j + 1] = guardado
    return lista


def ordenar_selecao_por_prioridade(lista):
    """Ordenação por seleção, prioridade CRESCENTE (exigida pela busca binária)."""
    for i in range(len(lista) - 1):
        indice_menor = i
        for j in range(i + 1, len(lista)):
            if lista[j]["prioridade"] < lista[indice_menor]["prioridade"]:
                indice_menor = j
        lista[i], lista[indice_menor] = lista[indice_menor], lista[i]
    return lista

# **4. Algoritmos de busca**

Duas buscas para localizar módulos rapidamente:

Busca sequencial: percorre item por item e encontra o módulo com menos combustível. Complexidade O(n).
Busca binária: encontra um módulo por prioridade, mas só funciona em lista já ordenada. Complexidade O(log n).

In [ ]:
def busca_sequencial_menor_combustivel(lista):
    """Percorre item por item e devolve o módulo com menos combustível. O(n)."""
    menor = lista[0]
    for item in lista:
        if item["combustivel"] < menor["combustivel"]:
            menor = item
    return menor


def busca_binaria_prioridade(lista_ordenada, alvo):
    """Busca módulo de prioridade 'alvo' em lista ORDENADA crescente. O(log n)."""
    inicio, fim = 0, len(lista_ordenada) - 1
    while inicio <= fim:
        meio = (inicio + fim) // 2
        if lista_ordenada[meio]["prioridade"] == alvo:
            return lista_ordenada[meio]
        elif lista_ordenada[meio]["prioridade"] < alvo:
            inicio = meio + 1
        else:
            fim = meio - 1
    return None

# **5. Regras lógicas de decisão**

Traduz as portas lógicas para código. A função avaliar aplica três regras booleanas:

Regra 1 (autorização): combustível OK AND área livre AND sensores OK AND NOT tempestade.
Regra 2 (alerta): NOT combustível OK OR NOT sensores OK.
Regra 3 (emergência): carga crítica AND NOT combustível OK.

O resultado pode ser EMERGÊNCIA, ALERTA, AUTORIZADO ou ADIADO.

In [ ]:
def avaliar(modulo, area_livre, tempestade, sensores_ok):
    """Aplica as 3 regras booleanas e devolve o resultado da análise."""
    C = modulo["combustivel"] >= LIMIAR_COMBUSTIVEL                  # combustível ok
    alerta = (not C) or (not sensores_ok)                            # Regra 2
    emergencia = (modulo["criticidade"] == "Crítica") and (not C)    # Regra 3

    if emergencia:
        return "EMERGÊNCIA"
    elif alerta:
        return "ALERTA"
    elif C and area_livre and sensores_ok and (not tempestade):      # Regra 1
        return "AUTORIZADO"
    else:
        return "ADIADO"

# **6. Função auxiliar de exibição**

Imprime a fila de pouso de forma legível, numerando as posições.

In [ ]:
def mostrar_fila(titulo, lista):
    print(titulo)
    for pos, m in enumerate(lista, start=1):
        print(f"  {pos}. {m['nome']:<15} prioridade={m['prioridade']}  "
              f"combustível={m['combustivel']}%  carga={m['criticidade']}")

# **7. Montagem da fila de pouso**

Ordena uma cópia da lista original por inserção e enfileira os módulos um a um com append, que entram no fim da fila. O primeiro da fila é o de maior prioridade.

In [ ]:
print("=== MGPEB - Aurora Siger ===\n")

for m in ordenar_insercao(modulos[:]):
    fila_pouso.append(m)                 # entra no fim da fila
mostrar_fila("Fila de pouso (ordenada por inserção):", fila_pouso)

# **8. Demonstração das buscas**

Aplica a busca sequencial na fila (menor combustível). Depois ordena uma cópia por prioridade crescente, com seleção, e aplica a busca binária para achar o módulo de prioridade 4.

In [2]:
menor = busca_sequencial_menor_combustivel(fila_pouso)
print(f"Busca sequencial -> menor combustível: {menor['nome']} ({menor['combustivel']}%)")

crescente = ordenar_selecao_por_prioridade(fila_pouso[:])
achado = busca_binaria_prioridade(crescente, 4)
print(f"Busca binária    -> módulo com prioridade 4: {achado['nome'] if achado else 'não encontrado'}")

# **9. Simulação do pouso em 2 rodadas**

Cada módulo sai do início da fila com pop(0) (FIFO), passa pela função avaliar e vai para a lista correspondente. Cada evento é empilhado no histórico.

In [ ]:
for rodada in (1, 2):
    tempestade = (rodada == 1)           # tempestade só na rodada 1
    area_livre = True
    sensores_ok = True
    clima = "TEMPESTADE" if tempestade else "céu limpo"
    print(f"\n--- Rodada {rodada} ({clima}) ---")

    while len(fila_pouso) > 0:
        modulo = fila_pouso.pop(0)       # sai do INÍCIO da fila (FIFO)
        resultado = avaliar(modulo, area_livre, tempestade, sensores_ok)

        if resultado == "AUTORIZADO":
            pousados.append(modulo)
            historico.append(f"Pouso autorizado: {modulo['nome']}")
        elif resultado == "ADIADO":
            em_espera.append(modulo)
            historico.append(f"Pouso adiado: {modulo['nome']}")
        else:  # ALERTA ou EMERGÊNCIA
            em_alerta.append(modulo)
            historico.append(f"{resultado}: {modulo['nome']}")
        print(f"  {modulo['nome']:<15} -> {resultado}")

    # Os adiados voltam para a fila, reordenados
    fila_pouso = ordenar_insercao(em_espera[:])
    em_espera.clear()

# **10. Uso da pilha: desfazer o último evento**

Mostra o funcionamento da pilha (LIFO). Consulta o topo com historico[-1] e o desempilha com pop(), simulando o “desfazer” do último evento registrado.

In [ ]:
print("Último evento (topo da pilha):", historico[-1])
removido = historico.pop()               # desempilha
print("Desfeito:", removido)

# **11. Resumo final**

Imprime o estado final das listas (pousados, em espera, em alerta) e o histórico de eventos, do mais antigo ao mais recente.

In [ ]:
print("=== Resumo final ===")
print("Pousados :", [m["nome"] for m in pousados])
print("Em espera:", [m["nome"] for m in em_espera])
print("Em alerta:", [m["nome"] for m in em_alerta])
print("Histórico (do mais antigo ao mais recente):")
for evento in historico:
    print("  -", evento)